In [ ]:
!git clone
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import tensorflow as tf
import keras
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.preprocessing import image
from tqdm import tqdm

# =========================
# CONFIG
# =========================
IMAGE_DIR = "/content/flickr30k/flickr30k-images"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/flickr30k_vgg16_features.npz"
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# =========================
# MODEL (VGG16 FEATURE EXTRACTOR)
# =========================
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

model = keras.Sequential([
    base_model,
    keras.layers.GlobalAveragePooling2D()  # -> (512,)
])

# =========================
# LOAD IMAGE PATHS
# =========================
image_paths = [
    os.path.join(IMAGE_DIR, fname)
    for fname in os.listdir(IMAGE_DIR)
    if fname.lower().endswith(".jpg")
]

print(f"Found {len(image_paths)} images")

# =========================
# FEATURE EXTRACTION (BATCHED)
# =========================
features = {}

for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):
    batch_paths = image_paths[i:i + BATCH_SIZE]

    batch_imgs = []
    batch_names = []

    for p in batch_paths:
        try:
            img = image.load_img(p, target_size=IMG_SIZE)
            x = image.img_to_array(img)
            batch_imgs.append(x)
            batch_names.append(os.path.basename(p))
        except Exception as e:
            print(f"Error loading {p}: {e}")

    if len(batch_imgs) == 0:
        continue

    batch_imgs = np.array(batch_imgs, dtype=np.float32)
    batch_imgs = preprocess_input(batch_imgs)

    batch_feats = model.predict(batch_imgs, verbose=0)

    for name, feat in zip(batch_names, batch_feats):
        features[name] = feat.astype(np.float32)

# =========================
# SAVE TO NPZ
# =========================
np.savez_compressed(OUTPUT_FILE, **features)

print(f"Saved features to {OUTPUT_FILE}")
print(f"Feature dimension: {next(iter(features.values())).shape}")